# CSD358 — Hand-labelling tool (standalone)
Rebuilds **exactly the same 80 sentences** as cell 5.2 of the main notebook from the files on your Drive, and continues
where you stopped (labels already given are kept). No GPU needed: **Runtime → Run all**, then label below.
When the counter shows 80/80, go back to the main notebook.

In [1]:
# ============================================================
# SETUP: load what the labelling tool needs from Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, html as _html
from IPython.display import display
ROOT = '/content/drive/MyDrive/CSD358_IR_Final'
DIRS = {k: f'{ROOT}/{k}' for k in ['data', 'results']}
SEED = 42
load_json = lambda p: json.load(open(p, encoding='utf-8'))
def save_json(obj, p):
    with open(p, 'w', encoding='utf-8') as f:
        json.dump(obj, f)
corpus = {}
with open(f"{DIRS['data']}/beir_scifact/corpus.jsonl", encoding='utf-8') as f:
    for line in f:
        r = json.loads(line); corpus[r['_id']] = {'title': r['title'], 'text': r['text']}
queries = {}
with open(f"{DIRS['data']}/beir_scifact/queries.jsonl", encoding='utf-8') as f:
    for line in f:
        r = json.loads(line); queries[r['_id']] = r['text']
test_qids = load_json(f"{DIRS['data']}/split_qids.json")['test_qids']
_sc = load_json(f"{DIRS['results']}/p4_scored_final.json")
scored = {'final': _sc['answers'] if 'answers' in _sc else _sc}
_lp = f"{DIRS['results']}/p5_hand_labels.json"
print(f"Loaded {len(corpus):,} abstracts, {len(test_qids)} test claims | labels saved so far: "
      f"{len(load_json(_lp)) if os.path.exists(_lp) else 0}")

Mounted at /content/drive
Loaded 5,183 abstracts, 300 test claims | labels saved so far: 42


In [2]:
# ============================================================
# LABELLING TOOL (identical to cell 5.2 of the main notebook)
# ============================================================
import ipywidgets as widgets
try:
    from google.colab import output as _colab_output
    _colab_output.enable_custom_widget_manager()
except Exception:
    pass

N_LABEL = 80
LABEL_PATH = f"{DIRS['results']}/p5_hand_labels.json"
pool = [(q, j) for q in test_qids for j, s in enumerate(scored['final'][q]['sentences']) if s['type'] == 'factual']
label_items = [f'{q}#{j}' for q, j in sorted(random.Random(SEED).sample(pool, min(N_LABEL, len(pool))))]
hand_labels = load_json(LABEL_PATH) if os.path.exists(LABEL_PATH) else {}

def _item_html(key):
    q, j = key.split('#'); j = int(j)
    s = scored['final'][q]['sentences'][j]
    cited = [d for d in s['cited'] if d in corpus]
    others = [d for d in scored['final'][q]['doc_ids'] if d not in cited]
    ab = lambda d: (f"<div style='margin:6px 0;padding:6px 8px;background:#f9f9f7;border-left:3px solid #2a78d6'>"
                    f"<b>[{d}] {_html.escape(corpus[d]['title'])}</b><br>{_html.escape(corpus[d]['text'])}</div>")
    return (f"<div style='font-family:sans-serif;font-size:13px;line-height:1.45;max-width:900px'>"
            f"<p><b>Claim:</b> {_html.escape(queries[q])}</p>"
            f"<p style='font-size:15px;background:#fff8e6;padding:8px'><b>Sentence to judge:</b> {_html.escape(s['text'])}</p>"
            f"<b>Cited abstract(s):</b> {''.join(ab(d) for d in cited) or '<i>none cited — see the context abstracts below</i>'}"
            f"<details {'open' if not cited else ''}><summary>Other context abstracts ({len(others)})</summary>"
            f"{''.join(ab(d) for d in others)}</details></div>")

_state = {'i': next((i for i, k in enumerate(label_items) if k not in hand_labels), len(label_items))}
_progress, _view = widgets.HTML(), widgets.HTML()
_buttons = [widgets.Button(description=t, layout=widgets.Layout(width='160px'), button_style=s)
            for t, s in [('Grounded', 'success'), ('Not supported', 'warning'), ('Contradicted', 'danger'), ('◀ Back', '')]]

def _render():
    i = _state['i']
    _progress.value = f"<b>{len(hand_labels)}/{len(label_items)} labelled</b>"
    _view.value = ("<h3>✅ All sentences labelled — run cell 5.3.</h3>" if i >= len(label_items) else
                   f"<p>Item {i + 1} of {len(label_items)}" + (f" · current label: <b>{hand_labels[label_items[i]]}</b>"
                   if label_items[i] in hand_labels else '') + "</p>" + _item_html(label_items[i]))

def _on_click(btn):
    if btn.description == '◀ Back':
        _state['i'] = max(0, _state['i'] - 1)
    elif _state['i'] < len(label_items):
        hand_labels[label_items[_state['i']]] = {'Grounded': 'grounded', 'Not supported': 'unsupported',
                                                'Contradicted': 'contradicted'}[btn.description]
        save_json(hand_labels, LABEL_PATH)
        _state['i'] += 1
    _render()

for b_ in _buttons:
    b_.on_click(_on_click)
_render()
display(widgets.VBox([_progress, widgets.HBox(_buttons), _view]))